# Train the CamDetec rodent model (free)

Trains a **YOLOv8** rodent detector on **your three Roboflow datasets**, merges
them into one clean single-class (`rodent`) set, and exports to **TensorFlow.js**
— producing a `model.json` that drops straight into CamDetec's pest-model slot.

Runs free on a **Colab T4 GPU** (Runtime → Change runtime type → **T4 GPU**).
Nothing to click in Roboflow — the notebook generates dataset versions itself.

**One-time setup:** add your Roboflow key as a Colab secret (next section).

In [1]:
%pip install -q ultralytics roboflow pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.1/42.1 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 33.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 276.9/276.9 kB 28.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 86.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 98.9 MB/s eta 0:00:00


## 1. GPU check

Should print `cuda … True`. If not: Runtime → Change runtime type → **T4 GPU**.

In [2]:
import torch
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (switch to a GPU runtime)")

torch 2.11.0+cu128 | cuda: True | Tesla T4


## 1b. Mount Google Drive (so a disconnect doesn't lose the run)

Colab's `/content` disk is **wiped the instant the runtime disconnects** — if your
laptop sleeps or the tab closes, an in-progress run on that disk is gone. Mounting
Drive sends every checkpoint to `MyDrive/camdetec/runs` instead, so a dropped
session **resumes** from the last epoch rather than restarting from scratch.

Approve the Google popup the first time. Set `USE_DRIVE = False` to fall back to
the old ephemeral behaviour.

In [ ]:
USE_DRIVE = True            # False = Colab disk only (lost on disconnect)
RUN_NAME  = "camdetec_rodent"

import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    RUN_PROJECT = "/content/drive/MyDrive/camdetec/runs"   # weights land here, safe across disconnects
    os.makedirs(RUN_PROJECT, exist_ok=True)
else:
    RUN_PROJECT = "/content/runs/detect"                    # ephemeral fallback

RUN_DIR = os.path.join(RUN_PROJECT, RUN_NAME)
print("Training output ->", RUN_DIR)
print("Drive-backed:", USE_DRIVE)

## 2. Roboflow API key (kept out of the notebook)

In Colab's left sidebar click the **🔑 key icon → Add new secret**, name it
`ROBOFLOW_API_KEY`, paste your **Private API Key** (app.roboflow.com → Settings →
API Keys), and enable notebook access. The key is never written into this file.

In [3]:
try:
    from google.colab import userdata
    ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
except Exception:
    ROBOFLOW_API_KEY = input('Roboflow Private API Key: ').strip()
assert ROBOFLOW_API_KEY, "no API key"
print("key loaded")

key loaded


## 3. Download datasets (your 3 + optional public)

Your three workspace datasets are auto-versioned at 640px. `INCLUDE_PUBLIC`
(below) also pulls 3 public Roboflow Universe rodent sets (~1600 more images,
incl. mice) for variety — set it to `False` to compare against your data alone.


In [ ]:
from roboflow import Roboflow

WORKSPACE = "rahils-workspace-jkbqh"
PROJECTS = [
    "rodent-cdscb-rc3pn",            # "Rodent"           ~818 imgs
    "rodent-detection-qodn4-19ued",  # "rodent detection" ~4980 imgs (cleanest)
    "rodent_detection-oljub",        # "Rodent_detection" ~4148 imgs
]
GEN_SETTINGS = {
    "augmentation": {},
    "preprocessing": {"auto-orient": True, "resize": {"width": 640, "height": 640, "format": "Stretch to"}},
}

# Optional public Roboflow Universe rodent datasets — ~1600 extra images with
# different cameras/lighting (and mice, not just rats) for better generalization.
# Set INCLUDE_PUBLIC = False to train on your own data only and compare mAP.
INCLUDE_PUBLIC = True
PUBLIC_DATASETS = [
    ("rodent-detection-6ctxr", "rodent-detection-and-repulsion", 1),  # ~968 rats
    ("winter-zebrafish-test", "did-rodent-yolov8", 1),                # ~260 mice
    ("markup", "rodent-6oqr3", 2),                                    # ~383 rodents
]

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset_dirs = []

# Your workspace datasets (auto-generate a 640 version if one doesn't exist).
for i, pid in enumerate(PROJECTS):
    proj = rf.workspace(WORKSPACE).project(pid)
    try:
        v = proj.generate_version(settings=GEN_SETTINGS)
    except Exception as e:
        print(f"[{pid}] generate_version failed ({e}); trying version 1")
        v = 1
    ds = proj.version(v).download("yolov8", location=f"/content/ds_{i}")
    dataset_dirs.append(ds.location)
    print(f"[{pid}] version {v} -> {ds.location}")

# Public datasets (download their existing versions; any class -> rodent in merge).
if INCLUDE_PUBLIC:
    for j, (ws, pid, ver) in enumerate(PUBLIC_DATASETS):
        try:
            ds = rf.workspace(ws).project(pid).version(ver).download("yolov8", location=f"/content/pub_{j}")
            dataset_dirs.append(ds.location)
            print(f"[public {ws}/{pid} v{ver}] -> {ds.location}")
        except Exception as e:
            print(f"[public {ws}/{pid}] skipped ({e})")

print("all datasets:", dataset_dirs)

## 4. Merge + collapse to a single `rodent` class

Your datasets have mixed/messy class names (a stray `0`, a leaked dataset-name
class). We're rodent-focused, so every box becomes one class: **rodent**. This
sidesteps all the label-name noise.

In [5]:
import os, glob, shutil, yaml

MERGED = "/content/rodent_merged"
SPLITS = ["train", "valid", "test"]
for s in SPLITS:
    os.makedirs(f"{MERGED}/{s}/images", exist_ok=True)
    os.makedirs(f"{MERGED}/{s}/labels", exist_ok=True)

counts = {s: 0 for s in SPLITS}
for i, loc in enumerate(dataset_dirs):
    for s in SPLITS:
        img_dir = f"{loc}/{s}/images"
        lbl_dir = f"{loc}/{s}/labels"
        if not os.path.isdir(img_dir):
            continue
        for img in os.listdir(img_dir):
            base, ext = os.path.splitext(img)
            new = f"d{i}_{base}"
            shutil.copy(f"{img_dir}/{img}", f"{MERGED}/{s}/images/{new}{ext}")
            lp = f"{lbl_dir}/{base}.txt"
            if os.path.exists(lp):
                out = []
                for ln in open(lp).read().strip().splitlines():
                    p = ln.split()
                    if len(p) >= 5:
                        p[0] = "0"          # single class -> rodent
                        out.append(" ".join(p))
                open(f"{MERGED}/{s}/labels/{new}.txt", "w").write("\n".join(out))
            counts[s] += 1

data = {"path": MERGED, "train": "train/images", "val": "valid/images",
        "test": "test/images", "nc": 1, "names": ["rodent"]}
yaml.safe_dump(data, open(f"{MERGED}/data.yaml", "w"))
DATA_YAML = f"{MERGED}/data.yaml"
print("merged image counts:", counts)
print("DATA_YAML:", DATA_YAML)

merged image counts: {'train': 6991, 'valid': 1965, 'test': 990}
DATA_YAML: /content/rodent_merged/data.yaml


## 5. Train

`yolov8n` (nano) — small enough to run in the browser. ~100 epochs on a T4 is
typically 30–90 min for ~10k images; early-stops if it plateaus.

In [ ]:
from ultralytics import YOLO
import os

# If an interrupted run left a checkpoint on Drive, resume it instead of starting
# over. YOLOv8 writes last.pt after EVERY epoch, so a disconnect costs one epoch.
last_ckpt = os.path.join(RUN_DIR, "weights", "last.pt")

if os.path.exists(last_ckpt):
    print("Found checkpoint — RESUMING from:", last_ckpt)
    model = YOLO(last_ckpt)
    model.train(resume=True)   # reuses epochs/imgsz/data from the checkpoint
else:
    print("Starting a fresh run ->", RUN_DIR)
    model = YOLO("yolov8n.pt")
    model.train(
        data=DATA_YAML,
        epochs=100,
        imgsz=640,          # MUST stay 640 — the app's YoloTfjsDetector expects imgsz=640
        batch=16,
        patience=25,
        project=RUN_PROJECT,    # <- saves to Drive when USE_DRIVE=True
        name=RUN_NAME,
        exist_ok=True,          # reuse the same run folder so resume stays predictable
    )

print("\nBest weights:", model.trainer.best)
print("Safe on Drive — a disconnect now means 'rerun this cell to resume', not 'restart'.")

## 6. Quick check — mAP50 above ~0.6 is a good rodent start.

In [ ]:
m = model.val()
print("mAP50:", round(float(m.box.map50), 3), "| mAP50-95:", round(float(m.box.map), 3))

## 7. Export to TensorFlow.js (imgsz 640, no embedded NMS — the app does NMS)

In [ ]:
export_path = model.export(format="tfjs", imgsz=640)
print("TF.js model at:", export_path)

## 8. Write labels.json + download the bundle

In [ ]:
import json, os, glob, shutil

web = str(export_path)
if not os.path.isdir(web):
    # Look in the run dir (Drive or local) for the exported _web_model folder.
    c = (glob.glob(os.path.join(RUN_DIR, "weights", "*_web_model"))
         or glob.glob("runs/detect/camdetec_rodent*/weights/*_web_model"))
    web = c[-1] if c else web
json.dump(["rodent"], open(os.path.join(web, "labels.json"), "w"))
print("web model:", web, "->", os.listdir(web))

shutil.make_archive("rodent_model", "zip", web)
try:
    from google.colab import files
    files.download("rodent_model.zip")
except Exception:
    print("Grab rodent_model.zip from the file browser.")

## 9. Install it in CamDetec

1. **Unzip** `rodent_model.zip` → `model.json`, `group1-shard*.bin`, `labels.json`.
2. Copy all into the repo at `public/models/pest/`.
3. Redeploy: `vercel --prod` (or commit + push).
4. In the app: **Settings → Object model → "Pest model · YOLOv8"**. It now detects
   `rodent`. The pest map, tracking, and trap suggestions all key off it.

**If accuracy is low:** dataset 3 (`rodent_detection-oljub`) has noisy labels —
try commenting it out of `PROJECTS` and retraining on the two clean sets. More
frames from your actual camera angle (and night/IR) help most. Retrain, re-export,
replace the files in `public/models/pest/`.